# Amazon SageMaker Multi-Model Endpoints using TorchServe


---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)

---

## Contents

With Amazon SageMaker multi-model endpoints, customers can create an endpoint that seamlessly hosts up to thousands of models. These endpoints are well suited to use cases where any one of many models, which can be served from a common inference container, needs to be called on-demand and where it is acceptable for infrequently invoked models to incur some additional latency. For applications which require consistently low inference latency, a traditional endpoint is still the best choice.

At a high level, Amazon SageMaker manages the loading and unloading of models for a multi-model endpoint, as they are needed. When an invocation request is made for a particular model, Amazon SageMaker routes the request to an instance assigned to that model, downloads the model artifacts from S3 onto that instance, and initiates loading of the model into the memory of the container. As soon as the loading is complete, Amazon SageMaker performs the requested invocation and returns the result. If the model is already loaded in memory on the selected instance, the downloading and loading steps are skipped, and the invocation is performed immediately.

This notebook uses SageMaker notebook instance conda_python3 kernel, demonstrates how to use TorchServe on SageMaker MME. In this example, there are 3 distinct models, each with its own set of dependencies, handler implementation and model configuration.

In [1]:
!python --version

zsh:1: command not found: python


In [2]:
# [exec-copy] pip installs disabled; env pre-provisioned (V3 + torch-model-archiver)

In [3]:
# Python Built-Ins:
from datetime import datetime
import os
import json
import logging
import time

# External Dependencies:
import boto3
from botocore.exceptions import ClientError

# V3: image_uris, the S3 upload helper, the session helper, the resource classes, and the shapes
# all live under sagemaker-core. There is no top-level `sagemaker` package in V3.
from sagemaker.core import image_uris
from sagemaker.core.s3 import S3Uploader
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.resources import Model, EndpointConfig, Endpoint
from sagemaker.core.shapes import ContainerDefinition, MultiModelConfig, ProductionVariant

sess = boto3.Session(region_name="us-west-2")  # [exec-copy] pinned region
sm = sess.client("sagemaker")
region = "us-west-2"  # [exec-copy] pinned (us-west-1 has no g5)
account = boto3.client("sts").get_caller_identity().get("Account")

smsess = Session(boto_session=sess)
role = "arn:aws:iam::729646638167:role/service-role/AmazonSageMaker-ExecutionRole-20251201T194045"  # [exec-copy] explicit execution role

# Configuration:
bucket_name = smsess.default_bucket()
prefix = "torchserve"
default_bucket_prefix = smsess.default_bucket_prefix

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"

output_path = f"s3://{bucket_name}/{prefix}/mme"
print(f"account={account}, region={region}, role={role}")

/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_data_url" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_data_source" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_description" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/

[07/16/26 13:53:54] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=7932548;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=7932549;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_version" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=7932554;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=7932555;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=7932560;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=7932561;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

account=729646638167, region=us-west-2, role=arn:aws:iam::729646638167:role/service-role/AmazonSageMaker-ExecutionRole-20251201T194045


## Create Model Artifacts
This example creates a TorchServe model artifact for each model.
### Install torch-model-archiver

In [4]:
# [exec-copy] pip installs disabled; env pre-provisioned (V3 + torch-model-archiver)

### Model 1: Segment Anything Model(SAM)
A new AI model from Meta that can segment any object in any image with a single click. No additional training needed. We are downloading one of the checkpoints
#### Download Segment Anything Model(SAM)

In [5]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

#### Implement customized handler
This step can be skipped if your model uses [TorchServe default handler](https://github.com/pytorch/serve/blob/ffa6847393cb7c36ae0122598152ca4614fe21f1/docs/default_handlers.md?plain=1#L1). Here we follow [TorchServe instruction](https://github.com/pytorch/serve/blob/ffa6847393cb7c36ae0122598152ca4614fe21f1/docs/custom_service.md?plain=1#L10) to create a customized handler for this model.

In [6]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

#### Config model

In [7]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

#### Custom dependencies

In [8]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

#### Create and upload `sam.tar.gz` file 

In [9]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

In [10]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

### Model 2: Stable Diffusion In Paint (SD)
#### Import and Save Stable Diffusion Model

In [11]:
# [exec-copy] pip installs disabled; env pre-provisioned (V3 + torch-model-archiver)

In [12]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

In [13]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

#### Implement customized handler
This step can be skipped if your model uses [TorchServe default handler](https://github.com/pytorch/serve/blob/ffa6847393cb7c36ae0122598152ca4614fe21f1/docs/default_handlers.md?plain=1#L1). Here we follow [TorchServe instruction](https://github.com/pytorch/serve/blob/ffa6847393cb7c36ae0122598152ca4614fe21f1/docs/custom_service.md?plain=1#L10) to create a customized handler for this model.

In [14]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

#### Config model

In [15]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

#### Custom dependencies

In [16]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

#### Create `sd.tar.gz` file

In [17]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

### Model 3: Large Mask In Painting Model (Lama)
#### Download Pre-Trained Model

In [18]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

#### Clone Lama Repo

In [19]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

#### Implement customized handler
This step can be skipped if your model uses [TorchServe default handler](https://github.com/pytorch/serve/blob/ffa6847393cb7c36ae0122598152ca4614fe21f1/docs/default_handlers.md?plain=1#L1). Here we follow [TorchServe instruction](https://github.com/pytorch/serve/blob/ffa6847393cb7c36ae0122598152ca4614fe21f1/docs/custom_service.md?plain=1#L10) to create a customized handler for this model.

In [20]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

#### Config model

In [21]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

#### Custom dependencies

In [22]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

#### Create `lama.tar.gz` file

In [23]:
# [exec-copy] reference-only: real SAM/SD/LaMa download+packaging skipped.
# Stand-in tars with identical S3 keys are staged in the deploy cell instead.
pass

## Create the Multi-Model Endpoint with the SageMaker SDK

### Create the Amazon SageMaker MultiDataModel entity

We create the multi-model endpoint using the [```MultiDataModel```](https://sagemaker.readthedocs.io/en/stable/api/inference/multi_data_model.html) class.

You can create a MultiDataModel by directly passing in a `sagemaker.model.Model` object - in which case, the Endpoint will inherit information about the image to use, as well as any environmental variables, network isolation, etc., once the MultiDataModel is deployed.

In addition, a MultiDataModel can also be created without explicitly passing a `sagemaker.model.Model` object. Please refer to the documentation for additional details.

In [24]:
# This is where our MME will read models from on S3.
multi_model_s3uri = output_path
print(multi_model_s3uri)

# Use SageMaker PyTorch Inference DLC
container = image_uris.retrieve(
    framework="pytorch",
    region=region,
    py_version="py310",
    image_scope="inference",
    version="2.2.0",
    instance_type="ml.m5.xlarge",  # [exec-copy] CPU DLC for stand-in
)
print(container)

# V3: instead of V2's MultiDataModel entity, we describe the multi-model container up front. The
# actual Model/EndpointConfig/Endpoint are created in the deploy cell below. Setting
# ContainerDefinition.mode="MultiModel" (plus multi_model_config) and pointing model_data_url at the
# S3 *prefix* (note the trailing "/", not a single tar) is exactly what V2's
# MultiDataModel(model=Model(...), model_data_prefix=...) did under the hood.
mme_container = ContainerDefinition(
    image=container,
    model_data_url=f"{multi_model_s3uri}/",
    mode="MultiModel",
    multi_model_config=MultiModelConfig(),
    environment={"TF_ENABLE_ONEDNN_OPTS": "0", "TS_INSTALL_PY_DEP_PER_MODEL": "true"},
)
print(mme_container)

s3://sagemaker-us-west-2-729646638167/torchserve/mme
763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-inference:2.2.0-cpu-py310
container_hostname=Unassigned() image='763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-inference:2.2.0-cpu-py310' image_config=Unassigned() mode='MultiModel' model_data_url='s3://sagemaker-us-west-2-729646638167/torchserve/mme/' model_data_source=Unassigned() additional_model_data_sources=Unassigned() environment={'TF_ENABLE_ONEDNN_OPTS': '0', 'TS_INSTALL_PY_DEP_PER_MODEL': 'true'} model_package_name=Unassigned() inference_specification_name=Unassigned() multi_model_config=MultiModelConfig(model_cache_setting=Unassigned())


### Deploy the Multi-Model Endpoint

You need to consider the appropriate instance type and number of instances for the projected prediction workload across all the models you plan to host behind your multi-model endpoint. The number and size of the individual models will also drive memory requirements.

In [25]:
# [exec-copy] stage the stand-in sam.tar.gz to the MME S3 prefix (mirrors the notebook,
# where only sam.tar.gz is present at first deploy).
S3Uploader.upload("/tmp/mme_stub-8b3f1a90/sam.tar.gz", multi_model_s3uri, sagemaker_session=smsess)

# V3: create the multi-model endpoint with sagemaker-core resources instead of MultiDataModel.deploy().
# Model (MultiModel mode) -> EndpointConfig (ProductionVariant) -> Endpoint, then wait for InService.
mme_name = "torchserve-mme-genai-" + datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
endpoint_config_name = mme_name + "-epc"
endpoint_name = mme_name + "-ep"

# Clean up a previous endpoint from a re-run, if any.
try:
    Endpoint.get(endpoint_name).delete()
    print("Deleting previous endpoint...")
    time.sleep(10)
except (NameError, ClientError):
    pass

Model.create(
    model_name=mme_name,
    primary_container=mme_container,
    execution_role_arn=role,
)

EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=mme_name,
            initial_instance_count=1,
            instance_type="ml.m5.xlarge",  # [exec-copy] CPU, avoids GPU capacity limits
            initial_variant_weight=1.0,
        )
    ],
)

predictor = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name,
)
predictor.wait_for_status("InService")

[07/16/26 13:53:56] WARNING  No region provided. Using default region.                                 ]8;id=7932568;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=7932569;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py#361\361]8;;\

                    INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=7932575;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=7932576;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py#375\375]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=7932581;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=7932582;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=7932587;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=7932588;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


...

                    INFO     Creating model resource.                                            ]8;id=7932595;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7932596;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#20593\20593]8;;\

[07/16/26 13:53:57] INFO     Creating endpoint_config resource.                                  ]8;id=7932602;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7932603;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#11069\11069]8;;\

[07/16/26 13:53:58] INFO     Creating endpoint resource.                                         ]8;id=7932609;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7932610;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10228\10228]8;;\

Output()

[07/16/26 13:57:07] INFO     Final Resource Status: InService                                    ]8;id=7932616;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7932617;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10484\10484]8;;\

### Our endpoint has launched! Let's look at what models are available to the endpoint!

By 'available', what we mean is, what model artifacts are currently stored under the S3 prefix we defined when setting up the `MultiDataModel` above i.e. `model_data_prefix`.

Currently, since we only have one artifact (i.e. `sam.tar.gz` files) stored under our defined S3 prefix.

In [26]:
# V3: "available" models are simply the *.tar.gz artifacts under our S3 prefix (same semantics as
# V2's MultiDataModel.list_models()). Right now only sam.tar.gz has been uploaded.
def list_mme_models(s3_prefix_uri):
    from urllib.parse import urlparse

    parsed = urlparse(s3_prefix_uri)
    bucket = parsed.netloc
    key_prefix = parsed.path.lstrip("/")
    if key_prefix and not key_prefix.endswith("/"):
        key_prefix += "/"
    s3 = boto3.client("s3")
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=key_prefix)
    return [
        obj["Key"][len(key_prefix):]
        for obj in resp.get("Contents", [])
        if obj["Key"].endswith(".tar.gz")
    ]


# Only sam.tar.gz visible!
list_mme_models(multi_model_s3uri)

['lama.tar.gz', 'sam.tar.gz', 'sd.tar.gz']

### Dynamically deploying models to the endpoint

The `.add_model()` method of the `MultiDataModel` will copy over our model artifacts from where they were initially stored, by training, to where our endpoint will source model artifacts for inference requests.

Note that we can continue using this method, as shown below, to dynamically deploy more models to our live endpoint as required!

`model_data_source` refers to the location of our model artifact (i.e. where it was deposited on S3 after training completed)

`model_data_path` is the **relative** path to the S3 prefix we specified above (i.e. `model_data_prefix`) where our endpoint will source models for inference requests. Since this is a **relative** path, we can simply pass the name of what we wish to call the model artifact at inference time.

In [27]:
# V3: MultiDataModel.add_model() just copies a local artifact into the endpoint's S3 prefix. We do the
# same with S3Uploader.upload; the live MME picks up new models dynamically on first invocation.
models = ["/tmp/mme_stub-8b3f1a90/sd.tar.gz", "/tmp/mme_stub-8b3f1a90/lama.tar.gz"]  # [exec-copy] stand-in tars
for model in models:
    S3Uploader.upload(model, multi_model_s3uri, sagemaker_session=smsess)

### Our models are ready to invoke!

We can see that the S3 prefix we specified when setting up `MultiDataModel` now has model artifacts listed. As such, the endpoint can now serve up inference requests for these models.

In [28]:
list_mme_models(multi_model_s3uri)

['lama.tar.gz', 'sam.tar.gz', 'sd.tar.gz']

## Get predictions from the endpoint

Recall that `mme.deploy()` returns a [Real Time Predictor](https://github.com/aws/sagemaker-python-sdk/blob/master/src/sagemaker/predictor.py#L35) that we saved in a variable called `predictor`.

That `predictor` can now be used as usual to request inference - but specifying which model to call:

In [29]:
# V3: the Predictor is replaced by a sagemaker-core Endpoint handle; invoke() targets a specific
# model via target_model, exactly like V2's predictor.predict(target_model=...).
predictor = Endpoint(endpoint_name=endpoint_name)
print(predictor)

endpoint_name='torchserve-mme-genai-2026-07-16-13-53-56-ep' endpoint_arn=Unassigned() endpoint_config_name=Unassigned() production_variants=Unassigned() data_capture_config=Unassigned() endpoint_status=Unassigned() failure_reason=Unassigned() creation_time=Unassigned() last_modified_time=Unassigned() last_deployment_config=Unassigned() async_inference_config=Unassigned() pending_deployment_summary=Unassigned() explainer_config=Unassigned() shadow_production_variants=Unassigned() metrics_config=Unassigned() serializer=None deserializer=None


### Model Segment Anything Inference Request

In [30]:
# sam payload
import base64
import json
import io
import numpy as np
from PIL import Image


def encode_image(img):
    # Convert the image to bytes
    with io.BytesIO() as output:
        img.save(output, format="JPEG")
        img_bytes = output.getvalue()

    return base64.b64encode(img_bytes).decode("utf-8")


img_file = "workspace/test_data/sample1.png"
img_bytes = None
with Image.open(img_file) as f:
    img_bytes = encode_image(f)

gen_args = json.dumps(dict(point_coords=[750, 500], point_labels=1, dilate_kernel_size=15))

payload = json.dumps({"image": img_bytes, "gen_args": gen_args}).encode("utf-8")

# V3: Endpoint.invoke replaces Predictor.predict; target_model selects the MME model.
# Note: use a relative key with no leading slash ("sam.tar.gz"). V2's MultiDataModel
# normalized a leading-slash name, but sagemaker-core forwards TargetModel verbatim and it is
# concatenated onto the model_data_url prefix, so a leading "/" would produce a double-slash
# key (mme//sam.tar.gz) that S3 cannot find.
response = predictor.invoke(
    body=payload,
    content_type="application/json",
    accept="application/json",
    target_model="sam.tar.gz",
)
encoded_masks_string = json.loads(response.body.read().decode("utf-8"))["generated_image"]
base64_bytes_masks = base64.b64decode(encoded_masks_string)

with Image.open(io.BytesIO(base64_bytes_masks)) as f:
    generated_image_rgb = f.convert("RGB")
    print('decoded image size:', generated_image_rgb.size)  # [exec-copy] headless

decoded image size: (64, 64)


### Model Stable Diffusion In Paint Inference Request

In [31]:
# sd payload
import base64
import json
import io
import numpy as np
from PIL import Image


def encode_image(img):
    # Convert the image to bytes
    with io.BytesIO() as output:
        img.save(output, format="JPEG")
        img_bytes = output.getvalue()

    return base64.b64encode(img_bytes).decode("utf-8")


img_file = "workspace/test_data/sample1.png"
img_bytes = None
with Image.open(img_file) as f:
    img_bytes = encode_image(f)

mask_file = "workspace/test_data/sample1_mask.jpg"
mask_bytes = None
with Image.open(mask_file) as f:
    mask_bytes = encode_image(f)

prompt = "a teddy bear on a bench"
nprompt = "ugly"
gen_args = json.dumps(dict(num_inference_steps=50, guidance_scale=10, seed=1))

payload = json.dumps(
    {
        "image": img_bytes,
        "mask_image": mask_bytes,
        "prompt": prompt,
        "negative_prompt": nprompt,
        "gen_args": gen_args,
    }
).encode("utf-8")

# V3: Endpoint.invoke replaces Predictor.predict; target_model selects the MME model.
# Note: use a relative key with no leading slash (see the SAM cell above).
response = predictor.invoke(
    body=payload,
    content_type="application/json",
    accept="application/json",
    target_model="sd.tar.gz",
)
encoded_masks_string = json.loads(response.body.read().decode("utf-8"))["generated_image"]
base64_bytes_masks = base64.b64decode(encoded_masks_string)
with Image.open(io.BytesIO(base64_bytes_masks)) as f:
    generated_image_rgb = f.convert("RGB")
    print('decoded image size:', generated_image_rgb.size)  # [exec-copy] headless

decoded image size: (64, 64)


### Large Mask In Painting Model Inference Request

In [32]:
# lama payload
import base64
import json
import io
import numpy as np
from PIL import Image


def encode_image(img):
    # Convert the image to bytes
    with io.BytesIO() as output:
        img.save(output, format="JPEG")
        img_bytes = output.getvalue()

    return base64.b64encode(img_bytes).decode("utf-8")


img_file = "workspace/test_data/sample1.png"
img_bytes = None
with Image.open(img_file) as f:
    img_bytes = encode_image(f)

mask_file = "workspace/test_data/sample1_mask.jpg"
mask_bytes = None
with Image.open(mask_file) as f:
    mask_bytes = encode_image(f)

payload = json.dumps(
    {
        "image": img_bytes,
        "mask_image": mask_bytes,
        "prompt": prompt,
        "negative_prompt": nprompt,
        "gen_args": gen_args,
    }
).encode("utf-8")

# V3: Endpoint.invoke replaces Predictor.predict; target_model selects the MME model.
# Note: use a relative key with no leading slash (see the SAM cell above).
response = predictor.invoke(
    body=payload,
    content_type="application/json",
    accept="application/json",
    target_model="lama.tar.gz",
)
encoded_masks_string = json.loads(response.body.read().decode("utf-8"))["generated_image"]
base64_bytes_masks = base64.b64decode(encoded_masks_string)
with Image.open(io.BytesIO(base64_bytes_masks)) as f:
    generated_image_rgb = f.convert("RGB")
    print('decoded image size:', generated_image_rgb.size)  # [exec-copy] headless

decoded image size: (64, 64)


## Updating a model

To update a model, you would follow the same approach as above and add it as a new model. For example, `ModelA-2`.

You should avoid overwriting model artifacts in Amazon S3, because the old version of the model might still be loaded in the endpoint's running container(s) or on the storage volume of instances on the endpoint: This would lead invocations to still use the old version of the model.

Alternatively, you could stop the endpoint and re-deploy a fresh set of models.

## Clean up

Endpoints should be deleted when no longer in use, since (per the [SageMaker pricing page](https://aws.amazon.com/sagemaker/pricing/)) they're billed by time deployed. Here we'll also delete the endpoint configuration - to keep things tidy.

In [33]:
# V3: delete endpoint, endpoint config, and model.
predictor.delete()
EndpointConfig.get(endpoint_config_name).delete()
Model.get(mme_name).delete()

[07/16/26 13:57:15] INFO     Deleting Endpoint - torchserve-mme-genai-2026-07-16-13-53-56-ep     ]8;id=7937029;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7937030;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10428\10428]8;;\

                    INFO     Deleting EndpointConfig -                                           ]8;id=7937036;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7937037;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#11220\11220]8;;\
                             torchserve-mme-genai-2026-07-16-13-53-56-epc                                          

                    INFO     Deleting Model - torchserve-mme-genai-2026-07-16-13-53-56           ]8;id=7937043;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7937044;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#20740\20740]8;;\

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/deploy_and_monitor|sm-mme_with_torchserve|sm-mme_with_torchserve.ipynb)